# Manifest centralizado de galaxias y cutouts RGB

Este notebook construye un registro maestro con una fila por galaxia de `data/morfologia_curada.csv`. Para cada campo de visión (FOV) descargado, agrega disponibilidad real del JPEG, ruta relativa al repositorio y la bandera `is_suspicious` calculada por `src.data.analyze_image_quality`.

El resultado no modifica el catálogo ni los índices de descarga. Se escribe como `data/processed/galaxy_manifest.csv` y queda listo para que los futuros datasets seleccionen una imagen por FOV sin depender del orden de archivos.

## Contrato del manifest

La llave de integración es `name`, por ejemplo `manga-10001-12701`. Para cada factor se generan columnas como:

- `fov_+2`: `True` solo si el índice reporta una descarga utilizable y el archivo JPEG existe.
- `fov_+2_path`: ruta relativa al JPEG, o valor ausente si no está disponible.
- `fov_+2_is_suspicious`: resultado de `analyze_image_quality(...)["is_suspicious"]`; queda ausente si la imagen no está disponible o si el análisis falla.

También se guardan `fov_+2_quality_status` y `fov_+2_quality_error` para distinguir un recorte inexistente de un problema al abrirlo. La bandera es una heurística para priorizar revisión, no un criterio automático de descarte.

In [4]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image

# El notebook puede ejecutarse desde VS Code o desde Jupyter sin depender del cwd.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = Path.cwd().resolve().parents[1]

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import analyze_image_quality

CATALOG_PATH = ROOT / "data/morfologia_curada.csv"
OUTPUT_PATH = ROOT / "data/galaxy_manifest.csv"
FOV_FACTORS = [+3, +2, +1, 0, -1, -2, -3]
SUCCESS_STATUSES = {"downloaded", "cached"}

FITS_DIR = ROOT / "data/MPL-11"

## 1. Funciones de integración

Cada índice se lee de forma independiente. Antes de incorporarlo se exige que `name` sea único: un merge muchos-a-uno ocultaría descargas duplicadas y rompería la correspondencia determinista entre galaxia y cutout.

La disponibilidad no se infiere solo de `status`: se comprueba además que la ruta exista en disco. Así, una fila `cached` con un archivo eliminado no se ofrecerá accidentalmente al entrenamiento.

In [5]:
def fov_label(factor: int) -> str:
    return f"{factor:+d}" if factor != 0 else "0"


def repository_relative_path(path_value: object) -> str | None:
    if pd.isna(path_value) or not str(path_value).strip():
        return None

    path = Path(str(path_value))
    absolute_path = path if path.is_absolute() else ROOT / path
    try:
        return absolute_path.resolve().relative_to(ROOT).as_posix()
    except ValueError:
        return absolute_path.resolve().as_posix()


def path_exists(path_value: str | None) -> bool:
    if path_value is None:
        return False
    path = Path(path_value)
    return (path if path.is_absolute() else ROOT / path).is_file()


def inspect_cutout(path_value: str) -> dict[str, object]:
    path = Path(path_value)
    absolute_path = path if path.is_absolute() else ROOT / path
    try:
        with Image.open(absolute_path) as source_image:
            rgb = np.asarray(source_image.convert("RGB"), dtype=np.float32) / 255.0
        report = analyze_image_quality(rgb)
        return {
            "is_suspicious": bool(report["is_suspicious"]),
            "quality_status": "analyzed",
            "quality_error": pd.NA,
        }
    except Exception as error:
        return {
            "is_suspicious": pd.NA,
            "quality_status": "error",
            "quality_error": str(error),
        }

In [6]:
def build_fov_columns(factor: int) -> pd.DataFrame:
    label = fov_label(factor)
    prefix = f"fov_{label}"
    index_path = ROOT / f"data/datasets/morfologia_curada_fov_{label}/download_index.csv"

    if not index_path.is_file():
        raise FileNotFoundError(f"No se encontró el índice de FOV {label}: {index_path}")

    index = pd.read_csv(index_path)
    required_columns = {"name", "local_path", "status"}
    missing_columns = required_columns - set(index.columns)
    if missing_columns:
        raise ValueError(f"{index_path} no contiene: {sorted(missing_columns)}")
    if index["name"].duplicated().any():
        duplicates = index.loc[index["name"].duplicated(keep=False), "name"].unique()
        raise ValueError(f"Índices duplicados en FOV {label}: {duplicates[:10].tolist()}")

    result = index[["name", "local_path", "status"]].copy()
    result[f"{prefix}_path"] = result["local_path"].map(repository_relative_path)
    result[prefix] = (
        result["status"].isin(SUCCESS_STATUSES)
        & result[f"{prefix}_path"].map(path_exists)
    )

    quality = [
        inspect_cutout(path)
        if available
        else {"is_suspicious": pd.NA, "quality_status": "not_available", "quality_error": pd.NA}
        for path, available in zip(result[f"{prefix}_path"], result[prefix], strict=True)
    ]
    quality_frame = pd.DataFrame(quality, index=result.index)
    result[f"{prefix}_is_suspicious"] = quality_frame["is_suspicious"].astype("boolean")
    result[f"{prefix}_quality_status"] = quality_frame["quality_status"]
    result[f"{prefix}_quality_error"] = quality_frame["quality_error"]

    return result.drop(columns=["local_path", "status"])

## 2. Construir la tabla maestra

El catálogo curado conserva sus columnas científicas y su etiqueta original. Los siete índices se unen por nombre, con una unión izquierda para asegurar que ninguna galaxia del catálogo desaparezca aunque todavía no tenga imágenes en alguno de los FOV.

Este paso puede tardar, porque abre cada JPEG disponible una vez para evaluar `is_suspicious`. No es necesario volver a calcularlo mientras los archivos y los thresholds de `analyze_image_quality` no cambien.

In [57]:
if not FITS_DIR.is_dir():
    raise FileNotFoundError(f"No se encontró el directorio FITS: {FITS_DIR}")

manifest = pd.read_csv(CATALOG_PATH)
if manifest["name"].duplicated().any():
    raise ValueError("El catálogo curado contiene IDs duplicados; resolverlos antes de crear el manifest.")

for factor in FOV_FACTORS:
    fov_index = build_fov_columns(factor)
    manifest = manifest.merge(fov_index, on="name", how="left", validate="one_to_one")

    label = fov_label(factor)
    prefix = f"fov_{label}"
    manifest[prefix] = manifest[prefix].fillna(False).astype(bool)

fits_records = []
for galaxy_name in manifest["name"]:
    fits_file = FITS_DIR / f"{galaxy_name}-1D.fits"
    available = fits_file.is_file()
    fits_records.append(
        {
            "fits_available": available,
            "fits_path": repository_relative_path(fits_file) if available else pd.NA,
        }
    )

manifest = pd.concat(
    [manifest, pd.DataFrame(fits_records, index=manifest.index)],
    axis=1,
)

manifest.head()

,name,objra,objdec,Type,Ttype,nsa_z,LogMass,incl,PETRO_TH90,label,fov_+3_path,fov_+3,fov_+3_is_suspicious,fov_+3_quality_status,fov_+3_quality_error,fov_+2_path,fov_+2,fov_+2_is_suspicious,fov_+2_quality_status,fov_+2_quality_error,fov_+1_path,fov_+1,fov_+1_is_suspicious,fov_+1_quality_status,fov_+1_quality_error,fov_0_path,fov_0,fov_0_is_suspicious,fov_0_quality_status,fov_0_quality_error,fov_-1_path,fov_-1,fov_-1_is_suspicious,fov_-1_quality_status,fov_-1_quality_error,fov_-2_path,fov_-2,fov_-2_is_suspicious,fov_-2_quality_status,fov_-2_quality_error,fov_-3_path,fov_-3,fov_-3_is_suspicious,fov_-3_quality_status,fov_-3_quality_error,fits_available,fits_path
0,manga-10001-12701,133.371091,57.598425,Sbc,4,0.039207,9.486857,70.407211,7.880948,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,False,analyzed,<NA>,True,data/MPL-11/manga-10001-12701-1D.fits
1,manga-10001-12702,133.685670,57.480250,SABbc,4,0.040811,9.727675,59.454862,14.147447,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,False,analyzed,<NA>,True,data/MPL-11/manga-10001-12702-1D.fits
2,manga-10001-12703,136.017160,57.092329,SABbc,4,0.046209,10.136514,78.126652,13.001850,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,True,analyzed,<NA>,True,data/MPL-11/manga-10001-12703-1D.fits
3,manga-10001-12704,133.989967,57.677968,Sd,7,0.014122,9.632117,81.373073,28.682878,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,False,analyzed,<NA>,True,data/MPL-11/manga-10001-12704-1D.fits
4,manga-10001-12705,136.751375,57.451437,Sbc,4,0.046235,10.113519,61.871475,11.039550,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,False,analyzed,<NA>,True,data/MPL-11/manga-10001-12705-1D.fits


In [58]:
manifest["fov_0_is_suspicious"].value_counts()

fov_0_is_suspicious
False    9636
True      370
Name: count, dtype: Int64

In [59]:
manifest["fov_-1_is_suspicious"].value_counts()

fov_-1_is_suspicious
False    9581
True      425
Name: count, dtype: Int64

In [60]:
manifest["fov_+1_is_suspicious"].value_counts()

fov_+1_is_suspicious
False    9627
True      379
Name: count, dtype: Int64

In [61]:
pd.options.display.max_columns = None

In [62]:
manifest

,name,objra,objdec,Type,Ttype,nsa_z,LogMass,incl,PETRO_TH90,label,fov_+3_path,fov_+3,fov_+3_is_suspicious,fov_+3_quality_status,fov_+3_quality_error,fov_+2_path,fov_+2,fov_+2_is_suspicious,fov_+2_quality_status,fov_+2_quality_error,fov_+1_path,fov_+1,fov_+1_is_suspicious,fov_+1_quality_status,fov_+1_quality_error,fov_0_path,fov_0,fov_0_is_suspicious,fov_0_quality_status,fov_0_quality_error,fov_-1_path,fov_-1,fov_-1_is_suspicious,fov_-1_quality_status,fov_-1_quality_error,fov_-2_path,fov_-2,fov_-2_is_suspicious,fov_-2_quality_status,fov_-2_quality_error,fov_-3_path,fov_-3,fov_-3_is_suspicious,fov_-3_quality_status,fov_-3_quality_error,fits_available,fits_path
0,manga-10001-12701,133.371091,57.598425,Sbc,4,0.039207,9.486857,70.407211,7.880948,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,False,analyzed,<NA>,True,data/MPL-11/manga-10001-12701-1D.fits
1,manga-10001-12702,133.685670,57.480250,SABbc,4,0.040811,9.727675,59.454862,14.147447,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,False,analyzed,<NA>,True,data/MPL-11/manga-10001-12702-1D.fits
2,manga-10001-12703,136.017160,57.092329,SABbc,4,0.046209,10.136514,78.126652,13.001850,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,True,analyzed,<NA>,True,data/MPL-11/manga-10001-12703-1D.fits
3,manga-10001-12704,133.989967,57.677968,Sd,7,0.014122,9.632117,81.373073,28.682878,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,False,analyzed,<NA>,True,data/MPL-11/manga-10001-12704-1D.fits
4,manga-10001-12705,136.751375,57.451437,Sbc,4,0.046235,10.113519,61.871475,11.039550,tardia,data/datasets/morfologia_curada_fov_+3/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_+1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_0/manga-10...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-1/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-2/manga-1...,True,False,analyzed,<NA>,data/datasets/morfologia_curada_fov_-3/manga-1...,True,False,analyzed,<NA>,True,data/MPL-11/manga-10001-12705-1D.fits
...,...,...,...,...,...,...,..

## 3. Resumen y persistencia

La tabla de resumen permite inspeccionar cobertura y alertas sin alterar ninguna fila. Finalmente, el CSV contiene rutas relativas para que pueda consumirse desde scripts ejecutados en la raíz del repositorio.

In [67]:
summary_records = []
for factor in FOV_FACTORS:
    label = fov_label(factor)
    prefix = f"fov_{label}"
    available = manifest[prefix]
    suspicious = manifest.loc[available, f"{prefix}_is_suspicious"]
    summary_records.append(
        {
            "fov": label,
            "available_images": int(available.sum()),
            "suspicious_images": int(suspicious.fillna(False).sum()),
            "quality_errors": int((manifest[f"{prefix}_quality_status"] == "error").sum()),
        }
    )

summary = pd.DataFrame(summary_records)
display(summary)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(OUTPUT_PATH, index=False)
print(f"Manifest guardado en: {OUTPUT_PATH}")
print(f"Filas: {len(manifest):,}; columnas: {len(manifest.columns):,}")

,fov,available_images,suspicious_images,quality_errors
0,+3,10006,418,0
1,+2,10006,387,0
2,+1,10006,379,0
3,0,10006,370,0
4,-1,10006,425,0
5,-2,10006,847,0
6,-3,10006,2227,0


Manifest guardado en: /Users/juliovelazquez/Documents/maestria/proyecto_integrador/clasificacion-galaxias/data/galaxy_manifest.csv
Filas: 10,014; columnas: 47


## Añadir validaciones de starlight

In [7]:
from astropy.io import fits

STARLIGHT_DIR = ROOT / "data/processed/starlight"
CHECK_FITS_REDSHIFT = True  # compara el Z de la cabecera de cada FITS con nsa_z (más lento)

STARLIGHT_INPUTS = {
    "starlight_spectrum": lambda name: STARLIGHT_DIR / "espectros_fits" / f"{name}-1D.fits",
    "starlight_populations": lambda name: STARLIGHT_DIR / "poblaciones_estelares" / f"{name}.npz",
    "starlight_properties": lambda name: STARLIGHT_DIR / "propiedades" / f"{name}.json",
}
STARLIGHT_COLUMNS = [
    column for prefix in STARLIGHT_INPUTS for column in (f"{prefix}_available", f"{prefix}_path")
]

if not STARLIGHT_DIR.is_dir():
    raise FileNotFoundError(f"No se encontró la salida de STARLIGHT: {STARLIGHT_DIR}")

manifest = pd.read_csv(OUTPUT_PATH)
if manifest["name"].duplicated().any():
    raise ValueError("El manifest contiene IDs duplicados.")
manifest = manifest.drop(columns=STARLIGHT_COLUMNS, errors="ignore")

records = []
for name in manifest["name"]:
    record = {}
    for prefix, build_path in STARLIGHT_INPUTS.items():
        file = build_path(name)
        available = file.is_file()
        record[f"{prefix}_available"] = available
        record[f"{prefix}_path"] = repository_relative_path(file) if available else pd.NA
    records.append(record)

starlight_columns = pd.DataFrame(records, index=manifest.index)
manifest = pd.concat([manifest, starlight_columns], axis=1)

# --- Validaciones ---
availability = manifest[[f"{prefix}_available" for prefix in STARLIGHT_INPUTS]]
partial = manifest.loc[availability.any(axis=1) & ~availability.all(axis=1), "name"]
if len(partial):
    raise ValueError(
        f"{len(partial)} galaxias tienen solo parte de los insumos STARLIGHT: {partial.head(10).tolist()}"
    )

known = set(manifest["name"])
for prefix, build_path in STARLIGHT_INPUTS.items():
    folder = build_path("x").parent
    suffix = build_path("x").name.removeprefix("x")
    on_disk = {p.name.removesuffix(suffix) for p in folder.glob(f"*{suffix}")} - {"indice", "base_poblaciones"}
    orphans = on_disk - known
    print(f"{prefix}: {len(on_disk):,} archivos; {len(orphans):,} sin fila en el manifest")

index_path = STARLIGHT_DIR / "propiedades" / "indice.csv"
if not index_path.is_file():
    raise FileNotFoundError(f"Falta {index_path}")
indexed = set(pd.read_csv(index_path)["name"])
ready = set(manifest.loc[manifest["starlight_spectrum_available"], "name"])
if indexed != ready | (indexed - known):
    missing_in_index = ready - indexed
    raise ValueError(f"indice.csv no coincide con los FITS del manifest (faltan {len(missing_in_index)}).")

if CHECK_FITS_REDSHIFT:
    mismatched = []
    rows = manifest.loc[manifest["starlight_spectrum_available"], ["name", "nsa_z", "starlight_spectrum_path"]]
    for row in rows.itertuples(index=False):
        header = fits.getheader(ROOT / row.starlight_spectrum_path, extname="Spectra")
        if not np.isclose(header["Z"], row.nsa_z, rtol=0, atol=1e-12):
            mismatched.append(row.name)
    if mismatched:
        raise ValueError(f"{len(mismatched)} FITS con Z distinto de nsa_z: {mismatched[:10]}")

# --- Guardado ---
summary = manifest[[f"{prefix}_available" for prefix in STARLIGHT_INPUTS]].sum()
display(summary.to_frame("galaxias"))
print(f"Sin insumos STARLIGHT: {(~availability.any(axis=1)).sum():,} de {len(manifest):,}")


starlight_spectrum: 10,013 archivos; 0 sin fila en el manifest
starlight_populations: 10,013 archivos; 0 sin fila en el manifest
starlight_properties: 10,013 archivos; 0 sin fila en el manifest


,galaxias
starlight_spectrum_available,10013
starlight_populations_available,10013
starlight_properties_available,10013


Sin insumos STARLIGHT: 1 de 10,014


In [8]:
manifest.to_csv(OUTPUT_PATH, index=False)
print(f"Manifest guardado en: {OUTPUT_PATH}")
print(f"Filas: {len(manifest):,}; columnas: {len(manifest.columns):,}")

Manifest guardado en: /Users/juliovelazquez/Documents/maestria/proyecto_integrador/clasificacion-galaxias/data/galaxy_manifest.csv
Filas: 10,014; columnas: 53


## 4. Splits persistentes para clasificación binaria

El split usa `label` como objetivo y `name` como identificador. Todas las variantes FOV de una galaxia permanecen en el mismo conjunto porque comparten una sola fila y un solo ID en el manifest.

La cohorte inicial de imágenes requiere disponibilidad y ausencia de la bandera `is_suspicious` en los FOV `+1`, `0` y `-1`. Una imagen sin análisis de calidad válido tampoco entra en esta cohorte. Esta regla define una muestra RGB limpia para el baseline; no implica que los demás FOV deban descartarse del manifest ni de futuros experimentos de sensibilidad.

La partición es 70% entrenamiento, 15% validación y 15% test, estratificada por `label` y con semilla fija. El notebook no sobrescribe un archivo existente sin cambiar explícitamente `OVERWRITE_SPLITS`.

In [72]:
from sklearn.model_selection import train_test_split

SUFIX = "_FoV"
SPLIT_SEED = 42
SPLITS_PATH = ROOT / "data/processed"
OBJECTIVE_COL = "label"

if SUFIX is None:
    SUFIX = ""

FILENAME = f"splits_seed{SPLIT_SEED}_obj-{OBJECTIVE_COL}{SUFIX}.csv"
FINAL_PATH = SPLITS_PATH / FILENAME
OVERWRITE_SPLITS = False

if SUFIX:
    CACHE_INDEX_PATH = ROOT / "data/processed/spectra" / f"spectra_cache{SUFIX if SUFIX else ''}" / "index.csv"

selected_fovs = ["fov_+1", "fov_0", "fov_-1"]
quality_columns = [f"{fov}_is_suspicious" for fov in selected_fovs]

if FINAL_PATH.exists() and not OVERWRITE_SPLITS:
    raise FileExistsError(
        f"{FINAL_PATH} ya existe. Se preserva como partición experimental; "
        "cambia OVERWRITE_SPLITS a True solo para regenerarlo deliberadamente."
    )

split_manifest = pd.read_csv(OUTPUT_PATH)
split_manifest = split_manifest[split_manifest["fits_available"].eq(True)].copy()

# Modificar respecto a cual vas a tomar
if SUFIX:
    cache_index = pd.read_csv(CACHE_INDEX_PATH)
    ready_ids = cache_index.query("status == 'ready'")["name"]
    split_manifest = manifest[manifest["name"].isin(ready_ids)].copy()

required_columns = set(selected_fovs) | set(quality_columns) | {"name", OBJECTIVE_COL}
missing_columns = required_columns - set(split_manifest.columns)
if missing_columns:
    raise ValueError(f"El manifest no contiene: {sorted(missing_columns)}")

quality_values = split_manifest[quality_columns]
usable_images = (
    split_manifest[selected_fovs].eq(True).all(axis=1)
    & quality_values.notna().all(axis=1)
    & quality_values.eq(False).all(axis=1)
)
split_source = split_manifest.loc[usable_images, ["name", OBJECTIVE_COL]].copy()

if split_source["name"].duplicated().any():
    raise ValueError("No se pueden crear splits: el manifest contiene IDs duplicados.")
if set(split_source[OBJECTIVE_COL].dropna()) != {"temprana", "tardia"}:
    raise ValueError("El split binario requiere exactamente las etiquetas 'temprana' y 'tardia'.")

train_names, held_out_names = train_test_split(
    split_source["name"],
    test_size=0.30,
    random_state=SPLIT_SEED,
    stratify=split_source[OBJECTIVE_COL],
)
labels_by_name = split_source.set_index("name")[OBJECTIVE_COL]
held_out_labels = labels_by_name.loc[held_out_names.to_numpy()]
validation_names, test_names = train_test_split(
    held_out_names,
    test_size=0.50,
    random_state=SPLIT_SEED,
    stratify=held_out_labels,
)

split_by_name = {
    **dict.fromkeys(train_names, "train"),
    **dict.fromkeys(validation_names, "validation"),
    **dict.fromkeys(test_names, "test"),
}
splits = split_source.assign(split=split_source["name"].map(split_by_name))

if splits["split"].isna().any() or splits["name"].duplicated().any():
    raise RuntimeError("La partición no asignó exactamente un split a cada galaxia.")

split_counts = (
    splits.groupby(["split", OBJECTIVE_COL], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "validation", "test"])
)
display(split_counts)

SPLITS_PATH.mkdir(parents=True, exist_ok=True)
splits.to_csv(FINAL_PATH, index=False)
print(f"Splits persistentes guardados en: {FINAL_PATH}")

label,tardia,temprana
split,,
train,4078,2467
validation,874,529
test,875,528


Splits persistentes guardados en: /Users/juliovelazquez/Documents/maestria/proyecto_integrador/clasificacion-galaxias/data/processed/splits_seed42_obj-label_FoV.csv
